## 1. 环境准备与数据加载

In [ ]:

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from scipy import stats
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print(f"  - Plotly 可用: {PLOTLY_AVAILABLE}")

In [ ]:
PROCESSED_DATA_DIR = "./processed_data" 
OUTPUT_DIR = "./output"             

# 原始数据路径（预处理时保存的副本）
RAW_DATA_PATH = os.path.join(PROCESSED_DATA_DIR, "raw_data.pkl")

# Top1000 异常交易 CSV 路径（直接读取完整数据）
TOP1000_CSV_PATH = os.path.join(OUTPUT_DIR, "top_1000_anomalies.csv")  

class ColumnIndex:
    uetr = 0                      # 交易唯一标识
    payment_channel = 1           # 交易渠道
    debit_bic_code = 2            # 付款行 BIC
    bene_bic_code = 3             # 收款行 BIC
    evt_tran_stat_cde = 4         # 交易状态码
    instructed_currency = 5       # 客户指定币种
    instructed_amount = 6         # 客户指定金额
    payment_currency = 7          # 银行使用币种
    payment_amount = 8            # 银行使用金额
    credit_currency = 9           # 收款方接收币种
    credit_amount = 10            # 收款方接收金额
    txn_dt = 11                   # 交易时间戳
    tds_dt = 12                   # 入库时间戳
    mop = 13                      # 付款方式
    debit_account_masked = 14     # 付款方账号（masked）
    bene_account_masked = 15      # 收款方账号（masked）

col_idx = ColumnIndex()

print("✓ 路径与列索引配置完成")
print(f"  - 原始数据: {RAW_DATA_PATH}")
print(f"  - Top1000 CSV: {TOP1000_CSV_PATH}")


In [ ]:
# ============================================================
# Cell 3: 加载原始数据（全量）
# ============================================================

print("正在加载原始数据...")

# 尝试从 pickle 加载（预处理时保存的副本）
if os.path.exists(RAW_DATA_PATH):
    df_all = pd.read_pickle(RAW_DATA_PATH)
    print(f"✓ 从 pickle 加载完成: {RAW_DATA_PATH}")
else:
    # 备用：如果 pickle 不存在，尝试从原始 CSV 加载
    # 请根据实际情况修改 CSV 路径
    csv_path = "../raw_data/xxx.csv"  # 根据你的实际路径调整
    if os.path.exists(csv_path):
        df_all = pd.read_csv(csv_path, header=0)
        print(f"✓ 从 CSV 加载完成: {csv_path}")
    else:
        raise FileNotFoundError(
            f"原始数据文件不存在！\n"
            f"  - 尝试路径 1: {RAW_DATA_PATH}\n"
            f"  - 尝试路径 2: {csv_path}\n"
            f"请检查路径配置或先运行预处理脚本 run_preprocess.py"
        )

print(f"\n原始数据 shape: {df_all.shape}")
print(f"列名（前5列）: {list(df_all.columns[:5])}")
print(f"\n数据预览:")
display(df_all.head(3))

In [ ]:
# ============================================================
# Cell 4: 加载 Top1000 异常交易（直接读取完整数据）
# ============================================================

print("正在加载 Top1000 异常交易...")

if not os.path.exists(TOP1000_CSV_PATH):
    raise FileNotFoundError(
        f"Top1000 CSV 文件不存在: {TOP1000_CSV_PATH}\n"
        f"请先运行训练脚本 run_training.py 生成异常检测结果"
    )

# 直接读取 Top1000 异常数据（字段格式与原始数据相同）
df_top1000 = pd.read_csv(TOP1000_CSV_PATH, header=0)

print(f"✓ Top1000 CSV 加载完成: {df_top1000.shape}")
print(f"列名: {list(df_top1000.columns)}")
print("\n数据预览:")
display(df_top1000.head(3))

print(f"\n✓ Top1000 异常交易已加载")
print(f"  共 {len(df_top1000)} 条异常交易")


In [ ]:
# ============================================================
# Cell 5: 创建对比数据集（全量 vs Top1000）
# ============================================================

# 添加标记列（方便后续分组）
df_all['is_top1000'] = False

# 通过 uetr 匹配 Top1000 交易在全量数据中的位置
if col_idx.uetr < df_all.shape[1] and col_idx.uetr < df_top1000.shape[1]:
    top1000_uetrs = set(df_top1000.iloc[:, col_idx.uetr].astype(str))
    df_all['is_top1000'] = df_all.iloc[:, col_idx.uetr].astype(str).isin(top1000_uetrs)
    matched_count = df_all['is_top1000'].sum()
    print(f"✓ 通过 uetr 匹配到 {matched_count} 条 Top1000 交易")
else:
    print("⚠ uetr 列不可用，无法进行匹配标记")

print(f"✓ 对比数据集创建完成")
print(f"  - 全量数据: {len(df_all):,} 条")
print(f"  - Top1000 异常: {len(df_top1000):,} 条")
print(f"  - Top1000 占比: {len(df_top1000)/len(df_all)*100:.4f}%")

# 快速验证：检查是否有缺失的关键字段
critical_cols = [col_idx.instructed_amount, col_idx.payment_amount, 
                col_idx.txn_dt, col_idx.debit_account_masked]
for col_i in critical_cols:
    if col_i < df_all.shape[1]:
        col_name = df_all.columns[col_i]
        missing_all = df_all.iloc[:, col_i].isnull().sum()
        missing_top = df_top1000.iloc[:, col_i].isnull().sum()
        print(f"  - {col_name}: 全量缺失 {missing_all}, Top1000缺失 {missing_top}")
    else:
        print(f"  ⚠ 列索引 {col_i} 超出范围")


## 2. 数值特征对比分析（金额、时延）

In [ ]:

numerical_features = {
    'instructed_amount': col_idx.instructed_amount,
    'payment_amount': col_idx.payment_amount,
    'credit_amount': col_idx.credit_amount
}

# 计算统计指标对比表
def compute_numerical_stats(df, df_subset, col_idx_dict):
    """
    计算数值特征的统计对比
    
    返回: DataFrame，包含全量与子集的分位数、均值、标准差等
    """
    stats_list = []
    
    for feat_name, col_i in col_idx_dict.items():
        if col_i >= df.shape[1]:
            continue
        
        # 提取列并转为数值
        all_vals = pd.to_numeric(df.iloc[:, col_i], errors='coerce').dropna()
        sub_vals = pd.to_numeric(df_subset.iloc[:, col_i], errors='coerce').dropna()
        
        # 计算统计量
        stats_list.append({
            'Feature': feat_name,
            'All_Count': len(all_vals),
            'All_Mean': all_vals.mean(),
            'All_Std': all_vals.std(),
            'All_P50': all_vals.quantile(0.5),
            'All_P95': all_vals.quantile(0.95),
            'All_P99': all_vals.quantile(0.99),
            'All_Max': all_vals.max(),
            'Top1000_Count': len(sub_vals),
            'Top1000_Mean': sub_vals.mean(),
            'Top1000_Std': sub_vals.std(),
            'Top1000_P50': sub_vals.quantile(0.5),
            'Top1000_P95': sub_vals.quantile(0.95),
            'Top1000_P99': sub_vals.quantile(0.99),
            'Top1000_Max': sub_vals.max(),
            # KS 统计量（衡量两分布差异）
            'KS_Statistic': stats.ks_2samp(all_vals, sub_vals).statistic,
            'KS_PValue': stats.ks_2samp(all_vals, sub_vals).pvalue
        })
    
    return pd.DataFrame(stats_list)

# 计算统计表
df_numerical_stats = compute_numerical_stats(df_all, df_top1000, numerical_features)

print("=" * 80)
print("数值特征统计对比表（金额）")
print("=" * 80)
display(df_numerical_stats)

# 解读提示
print("\n📊 解读提示:")
print("  - 如果 Top1000_P95/P99 >> All_P95/P99，说明异常集中在极端大额")
print("  - KS_Statistic 越大（>0.3），说明 Top1000 的分布越偏离全量")
print("  - 注意 Mean vs P50：如果 Mean >> P50，说明存在长尾（建议用对数尺度）")

In [ ]:

# 由于金额通常是长尾分布，使用对数尺度更清晰
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for i, (feat_name, col_i) in enumerate(numerical_features.items()):
    if col_i >= df_all.shape[1]:
        continue
    
    # 提取数据
    all_vals = pd.to_numeric(df_all.iloc[:, col_i], errors='coerce').dropna()
    top_vals = pd.to_numeric(df_top1000.iloc[:, col_i], errors='coerce').dropna()
    
    # 过滤掉 0 值（对数尺度不能处理 0）
    all_vals_log = np.log1p(all_vals[all_vals > 0])
    top_vals_log = np.log1p(top_vals[top_vals > 0])
    
    # 左侧：对数直方图
    ax = axes[i, 0]
    ax.hist(all_vals_log, bins=50, alpha=0.5, color='skyblue', 
           label=f'All (n={len(all_vals_log):,})', density=True)
    ax.hist(top_vals_log, bins=30, alpha=0.7, color='red', 
           label=f'Top1000 (n={len(top_vals_log):,})', density=True)
    ax.set_xlabel('log(Amount + 1)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{feat_name} - Log Distribution', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # 右侧：箱线图对比
    ax = axes[i, 1]
    bp = ax.boxplot([all_vals, top_vals], 
                     labels=['All', 'Top1000'],
                     patch_artist=True,
                     showfliers=False)  # 不显示离群点，避免图太挤
    bp['boxes'][0].set_facecolor('skyblue')
    bp['boxes'][1].set_facecolor('red')
    ax.set_ylabel('Amount', fontsize=11)
    ax.set_title(f'{feat_name} - Boxplot', fontsize=12, fontweight='bold')
    ax.set_yscale('log')  # 对数 Y 轴
    ax.grid(alpha=0.3, axis='y')
    
    # 添加统计注释
    sep = (top_vals.median() - all_vals.median()) / (all_vals.std() + 1e-8)
    ax.text(0.5, 0.95, f'Median Separation: {sep:.2f}σ', 
           transform=ax.transAxes, ha='center', va='top',
           bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.6))

plt.suptitle('数值特征分布对比：金额（对数尺度）', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ 金额分布图绘制完成")

In [ ]:

# 计算时延：tds_dt - txn_dt（入库时间 - 交易时间）
def compute_time_diff(df):
    """计算时延（秒）"""
    txn_dt = pd.to_datetime(df.iloc[:, col_idx.txn_dt], errors='coerce')
    tds_dt = pd.to_datetime(df.iloc[:, col_idx.tds_dt], errors='coerce')
    time_diff = (tds_dt - txn_dt).dt.total_seconds()
    return time_diff

time_diff_all = compute_time_diff(df_all)
time_diff_top = compute_time_diff(df_top1000)

# 统计对比
print("=" * 80)
print("时延特征统计对比（time_diff_seconds）")
print("=" * 80)

stats_dict = {
    'Metric': ['Count', 'Mean', 'Std', 'P50', 'P95', 'P99', 'Max', 'Min'],
    'All': [
        time_diff_all.count(),
        time_diff_all.mean(),
        time_diff_all.std(),
        time_diff_all.quantile(0.5),
        time_diff_all.quantile(0.95),
        time_diff_all.quantile(0.99),
        time_diff_all.max(),
        time_diff_all.min()
    ],
    'Top1000': [
        time_diff_top.count(),
        time_diff_top.mean(),
        time_diff_top.std(),
        time_diff_top.quantile(0.5),
        time_diff_top.quantile(0.95),
        time_diff_top.quantile(0.99),
        time_diff_top.max(),
        time_diff_top.min()
    ]
}

df_time_stats = pd.DataFrame(stats_dict)
display(df_time_stats)

# KS 检验
ks_stat, ks_p = stats.ks_2samp(time_diff_all.dropna(), time_diff_top.dropna())
print(f"\nKS Statistic: {ks_stat:.4f}, P-value: {ks_p:.4e}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左侧：直方图
ax = axes[0]
ax.hist(time_diff_all.dropna(), bins=50, alpha=0.5, color='skyblue', 
       label='All', density=True)
ax.hist(time_diff_top.dropna(), bins=30, alpha=0.7, color='red', 
       label='Top1000', density=True)
ax.set_xlabel('Time Diff (seconds)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Time Delay Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 右侧：箱线图
ax = axes[1]
bp = ax.boxplot([time_diff_all.dropna(), time_diff_top.dropna()], 
                 labels=['All', 'Top1000'],
                 patch_artist=True,
                 showfliers=False)
bp['boxes'][0].set_facecolor('skyblue')
bp['boxes'][1].set_facecolor('red')
ax.set_ylabel('Time Diff (seconds)', fontsize=11)
ax.set_title('Time Delay Comparison', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

plt.suptitle('时延特征对比', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 解读提示:")
print("  - 如果 Top1000 的时延 P95/P99 显著高于全量，可能是'延迟入库'的异常")
print("  - 如果 Top1000 的时延极小甚至负数，可能是时间戳异常")

## 3. 类别特征对比分析（渠道、银行、币种、方式）

In [ ]:

# 定义要分析的类别列
categorical_features = {
    'payment_channel': col_idx.payment_channel,
    'mop': col_idx.mop,
    'instructed_currency': col_idx.instructed_currency,
    'payment_currency': col_idx.payment_currency,
    'debit_bic_code': col_idx.debit_bic_code,
    'bene_bic_code': col_idx.bene_bic_code,
    'evt_tran_stat_cde': col_idx.evt_tran_stat_cde  # 状态码（重要但模型未用）
}

def compute_lift_table(df, df_subset, col_i, feat_name, top_n=20):
    """
    计算类别特征的 Lift（异常富集度）
    
    Lift = (该类别在 Top1000 中的占比) / (该类别在全量中的占比)
    Lift > 1: 异常富集；Lift < 1: 异常稀释
    """
    all_counts = df.iloc[:, col_i].astype(str).value_counts()
    sub_counts = df_subset.iloc[:, col_i].astype(str).value_counts()
    
    # 合并统计
    combined = pd.DataFrame({
        'All_Count': all_counts,
        'Top1000_Count': sub_counts
    }).fillna(0)
    
    combined['All_Pct'] = combined['All_Count'] / len(df) * 100
    combined['Top1000_Pct'] = combined['Top1000_Count'] / len(df_subset) * 100
    combined['Lift'] = combined['Top1000_Pct'] / (combined['All_Pct'] + 1e-8)
    
    # 按 Lift 降序排序，取前 N
    combined = combined.sort_values('Lift', ascending=False).head(top_n)
    combined = combined.reset_index()
    combined.rename(columns={'index': feat_name}, inplace=True)
    
    return combined

# 为每个类别特征计算 Lift 表
lift_tables = {}
for feat_name, col_i in categorical_features.items():
    if col_i >= df_all.shape[1]:
        print(f"⚠ 跳过 {feat_name}（列索引 {col_i} 超出范围）")
        continue
    
    lift_tables[feat_name] = compute_lift_table(df_all, df_top1000, col_i, feat_name, top_n=15)
    
    print(f"\n{'='*80}")
    print(f"{feat_name} - Top15 异常富集类别（按 Lift 排序）")
    print(f"{'='*80}")
    display(lift_tables[feat_name])

print("\n📊 解读提示:")
print("  - Lift > 2: 该类别在 Top1000 中的占比是全量的 2 倍以上，异常高度富集")
print("  - Lift < 0.5: 该类别在 Top1000 中反而稀少，可能是'正常类别'")
print("  - 关注 Top1000_Count 高 + Lift 高的类别：既常见又异常")

In [ ]:

# 选择几个关键类别特征绘制 Lift 条形图
key_features = ['payment_channel', 'mop', 'instructed_currency', 'evt_tran_stat_cde']
available_features = [f for f in key_features if f in lift_tables]

if len(available_features) == 0:
    print("⚠ 无可用的类别特征进行可视化")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for i, feat_name in enumerate(available_features[:4]):
        if i >= len(axes):
            break
        
        ax = axes[i]
        df_lift = lift_tables[feat_name].head(10)  # 取前 10
        
        # 绘制 Lift 条形图
        colors = ['red' if x > 1.5 else 'orange' if x > 1 else 'skyblue' 
                 for x in df_lift['Lift']]
        
        bars = ax.barh(range(len(df_lift)), df_lift['Lift'], color=colors, edgecolor='black')
        ax.set_yticks(range(len(df_lift)))
        ax.set_yticklabels(df_lift[feat_name], fontsize=9)
        ax.set_xlabel('Lift (Top1000% / All%)', fontsize=11)
        ax.set_title(f'{feat_name} - Anomaly Lift', fontsize=12, fontweight='bold')
        ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Baseline')
        ax.invert_yaxis()
        ax.legend()
        ax.grid(alpha=0.3, axis='x')
        
        # 添加数值标签
        for j, (bar, val, count) in enumerate(zip(bars, df_lift['Lift'], df_lift['Top1000_Count'])):
            ax.text(val + 0.05, j, f'{val:.2f} (n={int(count)})', 
                   va='center', fontsize=8)
    
    # 隐藏多余的子图
    for j in range(len(available_features), len(axes)):
        axes[j].axis('off')
    
    plt.suptitle('类别特征异常富集度（Lift）', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✓ Lift 可视化完成")

In [ ]:

# 示例：payment_channel × mop 的交叉占比热力图
# 这能发现"稀有组合"是否在 Top1000 中富集

def plot_category_heatmap(df, df_subset, col_i1, col_i2, name1, name2, top_n=10):
    """
    绘制两个类别特征的交叉占比热力图
    """
    # 提取类别
    cat1_all = df.iloc[:, col_i1].astype(str)
    cat2_all = df.iloc[:, col_i2].astype(str)
    cat1_top = df_subset.iloc[:, col_i1].astype(str)
    cat2_top = df_subset.iloc[:, col_i2].astype(str)
    
    # 选择 Top-N 最常见的类别（避免热力图太大）
    top_cat1 = cat1_all.value_counts().head(top_n).index.tolist()
    top_cat2 = cat2_all.value_counts().head(top_n).index.tolist()
    
    # 创建交叉表
    ct_all = pd.crosstab(cat1_all, cat2_all, normalize='all') * 100
    ct_top = pd.crosstab(cat1_top, cat2_top, normalize='all') * 100
    
    # 过滤到 Top-N
    ct_all = ct_all.loc[top_cat1, top_cat2]
    ct_top = ct_top.reindex(index=top_cat1, columns=top_cat2, fill_value=0)
    
    # Lift = Top1000% / All%
    ct_lift = ct_top / (ct_all + 1e-8)
    
    # 绘图
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # 左：全量占比
    sns.heatmap(ct_all, annot=True, fmt='.2f', cmap='Blues', 
               ax=axes[0], cbar_kws={'label': 'Percentage (%)'})
    axes[0].set_title(f'{name1} × {name2} - All Data', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(name2)
    axes[0].set_ylabel(name1)
    
    # 中：Top1000 占比
    sns.heatmap(ct_top, annot=True, fmt='.2f', cmap='Reds', 
               ax=axes[1], cbar_kws={'label': 'Percentage (%)'})
    axes[1].set_title(f'{name1} × {name2} - Top1000', fontsize=12, fontweight='bold')
    axes[1].set_xlabel(name2)
    axes[1].set_ylabel(name1)
    
    # 右：Lift
    sns.heatmap(ct_lift, annot=True, fmt='.2f', cmap='RdYlGn_r', 
               center=1.0, vmin=0, vmax=3,
               ax=axes[2], cbar_kws={'label': 'Lift'})
    axes[2].set_title(f'{name1} × {name2} - Lift', fontsize=12, fontweight='bold')
    axes[2].set_xlabel(name2)
    axes[2].set_ylabel(name1)
    
    plt.tight_layout()
    plt.show()

# 示例：payment_channel × mop
if col_idx.payment_channel < df_all.shape[1] and col_idx.mop < df_all.shape[1]:
    print("绘制 payment_channel × mop 交叉热力图...")
    plot_category_heatmap(df_all, df_top1000, 
                         col_idx.payment_channel, col_idx.mop,
                         'payment_channel', 'mop', top_n=8)
else:
    print("⚠ 无法绘制热力图（列索引超出范围）")

print("\n📊 解读提示:")
print("  - Lift 热力图中红色格子（Lift > 1.5）代表异常富集的组合")
print("  - 如果某些格子在 All 中几乎为 0，但在 Top1000 中出现，说明是'稀有异常组合'")

## 4. 时间维度对比分析（交易时段）

In [ ]:

# 从 txn_dt 提取时间特征
def extract_time_features(df):
    """从交易时间戳提取 hour 和 dayofweek"""
    txn_dt = pd.to_datetime(df.iloc[:, col_idx.txn_dt], errors='coerce')
    return {
        'hour': txn_dt.dt.hour,
        'dayofweek': txn_dt.dt.dayofweek,  # 0=Monday, 6=Sunday
        'date': txn_dt.dt.date
    }

time_feat_all = extract_time_features(df_all)
time_feat_top = extract_time_features(df_top1000)

# 小时分布对比
hour_dist_all = time_feat_all['hour'].value_counts(normalize=True).sort_index() * 100
hour_dist_top = time_feat_top['hour'].value_counts(normalize=True).sort_index() * 100

# 星期分布对比
dow_dist_all = time_feat_all['dayofweek'].value_counts(normalize=True).sort_index() * 100
dow_dist_top = time_feat_top['dayofweek'].value_counts(normalize=True).sort_index() * 100

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左：小时分布
ax = axes[0]
x = np.arange(24)
width = 0.35
ax.bar(x - width/2, hour_dist_all.reindex(x, fill_value=0), width, 
      label='All', color='skyblue', alpha=0.7)
ax.bar(x + width/2, hour_dist_top.reindex(x, fill_value=0), width, 
      label='Top1000', color='red', alpha=0.7)
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_title('Transaction Hour Distribution', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.legend()
ax.grid(alpha=0.3, axis='y')

# 右：星期分布
ax = axes[1]
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
x = np.arange(7)
ax.bar(x - width/2, dow_dist_all.reindex(x, fill_value=0), width, 
      label='All', color='skyblue', alpha=0.7)
ax.bar(x + width/2, dow_dist_top.reindex(x, fill_value=0), width, 
      label='Top1000', color='red', alpha=0.7)
ax.set_xlabel('Day of Week', fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_title('Transaction Day Distribution', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(dow_labels)
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.suptitle('时间维度分布对比', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 解读提示:")
print("  - 如果 Top1000 在夜间（23:00-05:00）占比显著高于全量，可能是'非工作时段异常'")
print("  - 如果 Top1000 在周末（Sat/Sun）占比异常，可能与业务规则有关")
print("  - 观察是否有某个小时/星期'断崖式'差异")

## 5. 网络结构维度对比（账户活跃度）

In [ ]:
# ============================================================
# Cell 13: 计算账户度数（基于全量数据）
# ============================================================

# 计算每个账户作为付款方/收款方的交易次数（度数）
def compute_account_degrees(df):
    """
    计算账户度数：
    - out_degree: 作为付款方的交易数
    - in_degree: 作为收款方的交易数
    """
    src_col = df.iloc[:, col_idx.debit_account_masked].astype(str)
    dst_col = df.iloc[:, col_idx.bene_account_masked].astype(str)
    
    out_degree = src_col.value_counts().to_dict()
    in_degree = dst_col.value_counts().to_dict()
    
    # 为每笔交易标记其付款方/收款方的度数
    df['src_out_degree'] = src_col.map(out_degree).fillna(0).astype(int)
    df['dst_in_degree'] = dst_col.map(in_degree).fillna(0).astype(int)
    df['min_degree'] = df[['src_out_degree', 'dst_in_degree']].min(axis=1)
    
    return df

# 计算全量数据的度数
df_all = compute_account_degrees(df_all)

# 为 Top1000 数据也添加度数（从全量数据的统计中映射）
src_col_top = df_top1000.iloc[:, col_idx.debit_account_masked].astype(str)
dst_col_top = df_top1000.iloc[:, col_idx.bene_account_masked].astype(str)

out_degree_dict = df_all.iloc[:, col_idx.debit_account_masked].astype(str).value_counts().to_dict()
in_degree_dict = df_all.iloc[:, col_idx.bene_account_masked].astype(str).value_counts().to_dict()

df_top1000['src_out_degree'] = src_col_top.map(out_degree_dict).fillna(0).astype(int)
df_top1000['dst_in_degree'] = dst_col_top.map(in_degree_dict).fillna(0).astype(int)
df_top1000['min_degree'] = df_top1000[['src_out_degree', 'dst_in_degree']].min(axis=1)

print("✓ 账户度数计算完成")
print(f"\n度数统计（全量）:")
print(f"  - src_out_degree: min={df_all['src_out_degree'].min()}, max={df_all['src_out_degree'].max()}, mean={df_all['src_out_degree'].mean():.2f}")
print(f"  - dst_in_degree: min={df_all['dst_in_degree'].min()}, max={df_all['dst_in_degree'].max()}, mean={df_all['dst_in_degree'].mean():.2f}")

print(f"\n度数统计（Top1000）:")
print(f"  - src_out_degree: min={df_top1000['src_out_degree'].min()}, max={df_top1000['src_out_degree'].max()}, mean={df_top1000['src_out_degree'].mean():.2f}")
print(f"  - dst_in_degree: min={df_top1000['dst_in_degree'].min()}, max={df_top1000['dst_in_degree'].max()}, mean={df_top1000['dst_in_degree'].mean():.2f}")


In [ ]:
# 可视化度数分布对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

degree_cols = ['src_out_degree', 'dst_in_degree', 'min_degree']
titles = ['Payer Out-Degree', 'Payee In-Degree', 'Min(Out, In) Degree']

for i, (col, title) in enumerate(zip(degree_cols, titles)):
    # 左列：对数直方图
    ax = axes[i, 0]
    
    # 过滤掉度数为 0（避免 log(0)）
    all_deg = df_all[col][df_all[col] > 0]
    top_deg = df_top1000[col][df_top1000[col] > 0]
    
    ax.hist(np.log1p(all_deg), bins=50, alpha=0.5, color='skyblue', 
           label='All', density=True)
    ax.hist(np.log1p(top_deg), bins=30, alpha=0.7, color='red', 
           label='Top1000', density=True)
    ax.set_xlabel(f'log({title} + 1)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'{title} - Log Distribution', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # 右列：箱线图
    ax = axes[i, 1]
    bp = ax.boxplot([all_deg, top_deg], 
                     labels=['All', 'Top1000'],
                     patch_artist=True,
                     showfliers=False)
    bp['boxes'][0].set_facecolor('skyblue')
    bp['boxes'][1].set_facecolor('red')
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(f'{title} - Comparison', fontsize=11, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(alpha=0.3, axis='y')

# 隐藏最后一个空子图
axes[-1, 0].axis('off')
axes[-1, 1].axis('off')

plt.suptitle('网络度数分布对比（对数尺度）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 解读提示:")
print("  - 如果 Top1000 的度数中位数 << 全量，说明异常主要来自'低活跃账户'（新账户/一次性）")
print("  - 如果 Top1000 的度数中位数 >> 全量，说明异常主要来自'高活跃账户'（疑似洗钱/中转）")
print("  - min_degree 是图模型门控融合的关键指标，观察其在 Top1000 中的分布")

In [ ]:
# ============================================================
# Cell 15: 账户集中度分析（Top1000 涉及的账户）
# ============================================================

# 统计 Top1000 涉及的账户
src_accounts_top = df_top1000.iloc[:, col_idx.debit_account_masked].astype(str)
dst_accounts_top = df_top1000.iloc[:, col_idx.bene_account_masked].astype(str)

src_account_counts = src_accounts_top.value_counts()
dst_account_counts = dst_accounts_top.value_counts()

print("=" * 80)
print("Top1000 涉及的账户集中度分析")
print("=" * 80)

print(f"\n付款方（Payer）账户:")
print(f"  - 唯一账户数: {src_account_counts.nunique()}")
print(f"  - Top10 账户贡献的交易数: {src_account_counts.head(10).sum()} / {len(df_top1000)} ({src_account_counts.head(10).sum()/len(df_top1000)*100:.2f}%)")
print(f"  - 最活跃账户的交易数: {src_account_counts.iloc[0]}")

print(f"\n收款方（Payee）账户:")
print(f"  - 唯一账户数: {dst_account_counts.nunique()}")
print(f"  - Top10 账户贡献的交易数: {dst_account_counts.head(10).sum()} / {len(df_top1000)} ({dst_account_counts.head(10).sum()/len(df_top1000)*100:.2f}%)")
print(f"  - 最活跃账户的交易数: {dst_account_counts.iloc[0]}")

# 可视化 Top20 账户
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左：付款方 Top20
ax = axes[0]
top20_src = src_account_counts.head(20)
ax.barh(range(len(top20_src)), top20_src.values, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(top20_src)))
ax.set_yticklabels([f"Acct_{i+1}" for i in range(len(top20_src))], fontsize=9)  # 匿名显示
ax.set_xlabel('Transaction Count in Top1000', fontsize=11)
ax.set_title('Top20 Payer Accounts in Top1000', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')

# 右：收款方 Top20
ax = axes[1]
top20_dst = dst_account_counts.head(20)
ax.barh(range(len(top20_dst)), top20_dst.values, color='coral', edgecolor='black')
ax.set_yticks(range(len(top20_dst)))
ax.set_yticklabels([f"Acct_{i+1}" for i in range(len(top20_dst))], fontsize=9)
ax.set_xlabel('Transaction Count in Top1000', fontsize=11)
ax.set_title('Top20 Payee Accounts in Top1000', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')

plt.suptitle('Top1000 账户集中度', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 解读提示:")
print("  - 如果 Top10 账户贡献超过 30%，说明异常高度集中在少数账户（可能是'异常团伙'）")
print("  - 如果账户数接近 1000（即几乎每笔交易都是不同账户），说明异常分散")


## 6. 综合对比总结与异常模式识别

In [ ]:

# Cell 16: 生成综合对比总结报告


print("=" * 80)
print("                   Top1000 异常交易特征对比总结报告")
print("=" * 80)

# 1. 基本统计
print("\n【1. 基本信息】")
print(f"  - 全量交易数: {len(df_all):,}")
print(f"  - Top1000 异常数: {len(df_top1000):,}")
print(f"  - 异常占比: {len(df_top1000)/len(df_all)*100:.4f}%")

# 2. 金额特征
print("\n【2. 金额特征】")
for feat_name in ['instructed_amount', 'payment_amount', 'credit_amount']:
    if feat_name in df_numerical_stats['Feature'].values:
        row = df_numerical_stats[df_numerical_stats['Feature'] == feat_name].iloc[0]
        print(f"\n  {feat_name}:")
        print(f"    - All P95/P99: {row['All_P95']:.2f} / {row['All_P99']:.2f}")
        print(f"    - Top1000 P95/P99: {row['Top1000_P95']:.2f} / {row['Top1000_P99']:.2f}")
        print(f"    - KS Statistic: {row['KS_Statistic']:.4f} {'(显著偏离)' if row['KS_Statistic'] > 0.3 else ''}")

# 3. 类别特征（Top3 富集类别）
print("\n【3. 类别特征（异常富集 Top3）】")
for feat_name in ['payment_channel', 'mop', 'evt_tran_stat_cde']:
    if feat_name in lift_tables:
        print(f"\n  {feat_name}:")
        top3 = lift_tables[feat_name].head(3)
        for _, row in top3.iterrows():
            cat = row[feat_name]
            lift = row['Lift']
            count = int(row['Top1000_Count'])
            print(f"    - {cat}: Lift={lift:.2f}, Count={count}")

# 4. 时间特征
print("\n【4. 时间特征】")
# 找出 Top1000 占比最高的 3 个小时
top3_hours = hour_dist_top.nlargest(3)
print(f"  Top1000 最集中的 3 个小时: {list(top3_hours.index)} ({top3_hours.values}%)")

# 找出 Top1000 占比最高的星期
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
top_dow = dow_dist_top.idxmax()
print(f"  Top1000 最集中的星期: {dow_names.get(top_dow, top_dow)} ({dow_dist_top.max():.2f}%)")

# 5. 网络结构
print("\n【5. 网络结构】")
print(f"  付款方账户唯一数: All={df_all.iloc[:, col_idx.debit_account_masked].nunique():,}, Top1000={src_account_counts.nunique()}")
print(f"  收款方账户唯一数: All={df_all.iloc[:, col_idx.bene_account_masked].nunique():,}, Top1000={dst_account_counts.nunique()}")
print(f"  Top10 付款方账户贡献占比: {src_account_counts.head(10).sum()/len(df_top1000)*100:.2f}%")
print(f"  Top10 收款方账户贡献占比: {dst_account_counts.head(10).sum()/len(df_top1000)*100:.2f}%")
print(f"  平均度数: All={df_all['min_degree'].mean():.2f}, Top1000={df_top1000['min_degree'].mean():.2f}")

print("\n" + "=" * 80)
print("✓ 总结报告生成完成")
print("=" * 80)

In [ ]:
# ============================================================
# Cell 17: 导出 Top1000 明细（含原始字段 + 图模型度数特征）
# ============================================================

# 首先需要为 Top1000 数据计算度数特征（从全量数据中获取）
print("正在为 Top1000 计算网络度数特征...")

# 从全量数据中获取度数统计
src_col_all = df_all.iloc[:, col_idx.debit_account_masked].astype(str)
dst_col_all = df_all.iloc[:, col_idx.bene_account_masked].astype(str)

out_degree_all = src_col_all.value_counts().to_dict()
in_degree_all = dst_col_all.value_counts().to_dict()

# 为 Top1000 每笔交易标记其度数
src_col_top = df_top1000.iloc[:, col_idx.debit_account_masked].astype(str)
dst_col_top = df_top1000.iloc[:, col_idx.bene_account_masked].astype(str)

df_top1000['src_out_degree'] = src_col_top.map(out_degree_all).fillna(0).astype(int)
df_top1000['dst_in_degree'] = dst_col_top.map(in_degree_all).fillna(0).astype(int)
df_top1000['min_degree'] = df_top1000[['src_out_degree', 'dst_in_degree']].min(axis=1)

print(f"✓ 度数特征计算完成")

# 选择关键列导出
output_cols = []
col_names = []

# 交易标识
if col_idx.uetr < df_top1000.shape[1]:
    output_cols.append(col_idx.uetr)
    col_names.append('uetr')

# 金额
for name, idx in [('instructed_amount', col_idx.instructed_amount),
                  ('payment_amount', col_idx.payment_amount),
                  ('credit_amount', col_idx.credit_amount)]:
    if idx < df_top1000.shape[1]:
        output_cols.append(idx)
        col_names.append(name)

# 账户
for name, idx in [('debit_account', col_idx.debit_account_masked),
                  ('bene_account', col_idx.bene_account_masked)]:
    if idx < df_top1000.shape[1]:
        output_cols.append(idx)
        col_names.append(name)

# 类别
for name, idx in [('payment_channel', col_idx.payment_channel),
                  ('mop', col_idx.mop),
                  ('debit_bic', col_idx.debit_bic_code),
                  ('bene_bic', col_idx.bene_bic_code),
                  ('status_code', col_idx.evt_tran_stat_cde)]:
    if idx < df_top1000.shape[1]:
        output_cols.append(idx)
        col_names.append(name)

# 时间
for name, idx in [('txn_dt', col_idx.txn_dt), ('tds_dt', col_idx.tds_dt)]:
    if idx < df_top1000.shape[1]:
        output_cols.append(idx)
        col_names.append(name)

# 创建导出表
df_export = df_top1000.iloc[:, output_cols].copy()
df_export.columns = col_names

# 添加度数特征
df_export['src_out_degree'] = df_top1000['src_out_degree'].values
df_export['dst_in_degree'] = df_top1000['dst_in_degree'].values
df_export['min_degree'] = df_top1000['min_degree'].values

# 导出
output_path = os.path.join(OUTPUT_DIR, "top1000_detailed.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_export.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✓ Top1000 明细已导出: {output_path}")
print(f"  - 行数: {len(df_export)}")
print(f"  - 列数: {len(df_export.columns)}")
print(f"  - 列名: {list(df_export.columns)}")
print("\n前 5 行预览:")
display(df_export.head())


## 7. （可选）高级分析：网络子图可视化

In [ ]:
# ============================================================
# Cell 18: Top1000 涉及账户的网络子图可视化（需要 networkx）
# ============================================================

try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except ImportError:
    NETWORKX_AVAILABLE = False
    print("⚠ NetworkX 未安装，跳过网络子图可视化")
    print("  安装命令: pip install networkx")

if NETWORKX_AVAILABLE:
    # 构建 Top1000 的交易网络
    G = nx.DiGraph()
    
    for _, row in df_top1000.iterrows():
        src = str(row.iloc[col_idx.debit_account_masked])
        dst = str(row.iloc[col_idx.bene_account_masked])
        amount = row.iloc[col_idx.payment_amount] if col_idx.payment_amount < len(row) else 1
        
        # 添加边（如果已存在则累加权重）
        if G.has_edge(src, dst):
            G[src][dst]['weight'] += 1
            G[src][dst]['total_amount'] += amount
        else:
            G.add_edge(src, dst, weight=1, total_amount=amount)
    
    print(f"✓ Top1000 网络图构建完成")
    print(f"  - 节点数: {G.number_of_nodes()}")
    print(f"  - 边数: {G.number_of_edges()}")
    
    # 如果节点太多，只可视化度数最高的子图
    if G.number_of_nodes() > 50:
        # 选择度数最高的 30 个节点
        degrees = dict(G.degree())
        top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:30]
        G_sub = G.subgraph(top_nodes).copy()
        print(f"  - 可视化子图（Top30 度数节点）: {G_sub.number_of_nodes()} 节点, {G_sub.number_of_edges()} 边")
    else:
        G_sub = G
    
    # 绘制网络图
    plt.figure(figsize=(14, 10))
    
    # 布局
    pos = nx.spring_layout(G_sub, k=0.5, iterations=50, seed=42)
    
    # 节点大小：按度数
    node_sizes = [G_sub.degree(n) * 100 for n in G_sub.nodes()]
    
    # 边宽度：按交易次数
    edge_widths = [G_sub[u][v]['weight'] * 0.5 for u, v in G_sub.edges()]
    
    # 绘制
    nx.draw_networkx_nodes(G_sub, pos, node_size=node_sizes, 
                          node_color='red', alpha=0.7, edgecolors='black')
    nx.draw_networkx_edges(G_sub, pos, width=edge_widths, 
                          alpha=0.5, edge_color='gray', arrows=True, 
                          arrowsize=15, arrowstyle='->')
    nx.draw_networkx_labels(G_sub, pos, font_size=8, font_color='white', 
                           font_weight='bold')
    
    plt.title('Top1000 异常交易网络子图\n(节点大小=度数, 边宽度=交易次数)', 
             fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print("\n📊 解读提示:")
    print("  - 密集连接的节点簇可能是'异常团伙'或'中转账户'")
    print("  - 孤立的节点对（只有一条边）可能是'一次性异常交易'")
    print("  - 星型结构（一个中心连多个外围）可能是'洗钱中转'")

---

## 分析完成总结

### 本 Notebook 完成的对比分析：

1. **数值特征（金额、时延）**
   - 分位数对比表（P50/P95/P99）
   - KS 检验（分布差异度）
   - 对数尺度直方图 + 箱线图

2. **类别特征（渠道、银行、币种、方式、状态）**
   - Lift 分析（异常富集度排行）
   - Top-N 类别占比对比
   - 交叉维度热力图（如 channel × mop）

3. **时间特征（小时、星期）**
   - 24小时分布对比
   - 星期分布对比

4. **网络结构（账户度数、集中度）**
   - 度数分布对比（out/in/min）
   - Top1000 涉及账户的集中度
   - 网络子图可视化（可选）

5. **综合总结报告**
   - 自动生成的对比摘要
   - Top1000 明细导出（含原始字段 + 度数 + 分数）

### 后续建议：

- 如果发现某些维度的 Lift 或 KS 异常高，可以深入分析该维度的"异常子模式"
- 可以基于本 Notebook 的发现，改进特征工程（例如把 `evt_tran_stat_cde` 加入模型）
- 网络子图中发现的"异常团伙"可以作为人工审核的优先级

---